## Aufbau eines Trainerskripts mit zwei Klassen:
- Trainer
- RunManager

In [3]:
#==Importe==
#Extern
import torch
from torch.utils.data import DataLoader, Subset
import torch.nn as nn
import matplotlib.pyplot as plt
import copy
#Intern
from aguirre.denoiser import DenoiseTransformer
from aguirre.diffusion import DiffusionProcess
from aguirre.config import Config, RUNS_DIR, PROJECT_ROOT, CACHE_DIR
from aguirre.data import Data_coordinator


cfg = Config()

In [ ]:
# TODO
# EMA in __Init__ integrieren. mit hardcopy
# Tokenloss in den Return


#Eval:
# Loss über EMA, dann Tokenloss bauen
# Loss über t: Sammelt Losses pro t über alle Batches und baut einen tensor mit der Shape (t, Epochs) auf.
# Konkret: mittlerer Loss pro Sequenz für timestep = t -> in den container für t. Mittelt den Container t
# Soll später Loss 3D-Graph zurückgeben mit t 0-1000 und pro t den mittleren Loss über Zeit
# wichtig: Wir brauchen festes t-Gitter für 


class Trainer():
    def __init__(
            self,
            optimizer: torch.optim,
            Model: DenoiseTransformer,
            Train_Data: DataLoader,
            Val_Data: DataLoader,
            cfg: Config = cfg,

    ):
        # EMA-Anlegen
        EMA_Model = copy.deepcopy(Model)
        EMA_Model = EMA_Model.requires_grad_(False)
        


        # Instanzen
        self.EMA_Model = EMA_Model
        self.Model = Model
        self.cfg = cfg
        self.Scheduler_train = DiffusionProcess(model = self.Model, cfg = self.cfg)
        self.Scheduler_val = DiffusionProcess(model = self.EMA_Model, cfg = self.cfg)
        self.ema_weight = (1 - cfg.ema_decay)
        self.optimizer = optimizer
        self.Train_Data = Train_Data.to(device = cfg.device)
        self.Val_Data = Val_Data.to(device = cfg.device)


    def forward_train(self, x, padding_mask, scheduler: DiffusionProcess, mode: str | None):
        x = self.Train_Data
        loss = self.Scheduler_train.forward(x_start = x,padding_mask= padding_mask, mode= mode)
        return loss

    def forward_val(self, x, padding_mask, scheduler: DiffusionProcess, mode: str | None):
        x = self.Train_Data
        loss = self.Scheduler_val.forward(x_start = x,padding_mask= padding_mask, mode= mode)
        return loss
    # Hier die Loss-funktion einmal mit etablieren: mit t über ein festgelegtes Raster. Vorher KI fragen, ob hier etablieren oder in diffusion.
    
    def update_EMA(self):
        with torch.no_grad():
            for theta, theta_EMA in zip(self.Model.parameters(), self.EMA_Model.parameters()):
                theta_EMA.lerp_(end = theta,weight = self.ema_weight)

    def train(self, token_loss: bool = True):
        # Training über eine Epoch
        self.Model.train()
        total_loss, count = 0.0 , 0

        for batch in self.Train_Data:
            x, p_mask = batch
            optim = self.optimizer
            optim.zero_grad()
            loss = self.forward_train(x_train= x, padding_mask= p_mask, scheduler= self.Scheduler, mode = self.cfg.loss_weight_scheme)
            if token_loss == True:
                tok_loss = self.forward_train(x_train= x, padding_mask= p_mask, scheduler= self.Scheduler, mode = "token")
            loss.backward()
            optim.step()
            self.update_EMA()

            total_loss += loss.item()
            total_tok_loss += tok_loss.item()
            count += 1

        if token_loss == True:
            tok_loss_epoch = total_tok_loss / count
            loss_epoch = total_loss/count
            return loss_epoch, tok_loss_epoch
        else:
            loss_epoch = total_loss/count
            return loss_epoch



    def evaluate(self):
        # Evaluation über eine Epoch
        self.Model.train()
        total_loss, count = 0.0 , 0
        
        for batch in self.Val_Data:
            x, p_mask = batch
        
    # Val-Loss
    # Val-token-loss
    # Val-t-loss  

    def summarize_eval(self):
        pass
    # Bau von Loss-Graphen:
    # Train-Loss
    # token-train_loss
    # val_loss über EMA
    # t_loss als 3D-Graph

    def print_report(self):
        pass
        # Baut aus der Config und "summarize_eval" ein PDF-Report


class RunManager():
    def __init__(self):
        pass

    def save_run(self):
        pass

    def load_run(self):
        pass

In [ ]:
#Dummy_modell Linearer Denoiser

class LinearDenoiser(nn.Module):
    """Wegwerf-Denoiser: ignoriert t, mappt jede Position einzeln."""

    def __init__(self, d: int):
        super().__init__()
        self.layer = nn.Linear(d, d)

    def forward(self, x):
        return self.layer(x)                 # (B, L, D)

Werner_Herzog = LinearDenoiser(d = cfg.embed_dim)

In [ ]:
#==Daten Laden==
#Trainer01 = Trainer()
train_ds = Data_coordinator(split = "train", cfg = cfg, standardize= True) 
val_ds = Data_coordinator(split= "val", cfg= cfg, standardize= True)

def small(ds, n):
    g = torch.Generator().manual_seed(cfg.seed)
    idx = torch.randperm(len(ds), generator=g)[:n].tolist()
    return Subset(ds, idx)

DEBUG = True
if DEBUG:
    train_ds = small(train_ds, 200)
    val_ds   = small(val_ds, 50)           # Val und Test ebenfalls verkleinern, sonst dauert die Eval länger als das Training
    #test_ds  = small(test_ds, 50)

train_d = DataLoader(dataset= train_ds, batch_size= cfg.batch_size, shuffle= True)
val_d = DataLoader(dataset= val_ds,batch_size= cfg.batch_size, shuffle= False)

In [ ]:
print(train_ds)